# STRATA, precisão sob demanda, E5: a mistura de níveis passa a escada do Hub? (Llama-3.1-8B)

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta**. Depois *Executar tudo*. Leva umas 3 horas; no fim, baixa dois arquivos (`.md` e `.json`) para colar na thread.

**A pergunta (E5 de `docs/08-precisao-sob-demanda.md`).** No modo "cabe no que tem", cada matriz das camadas fica no nível 0, 1 ou 2 até encher a RAM disponível, a partir do Q4_K_M com imatrix. O Hub já tem 7 degraus nessa faixa (bartowski: Q4_K_M, Q4_K_L, Q5_K_S, Q5_K_M, Q5_K_L, Q6_K, Q6_K_L, de 4,58 a 6,38 GiB). A mistura só tem valor de qualidade se, em algum tamanho, tiver KL menor que **o melhor arquivo da escada que cabe** naquele tamanho.

**Regra escrita antes de rodar:**
- **Sucesso:** a mistura (tipos P ou D) vence o melhor degrau que cabe numa faixa contínua de pelo menos 0,2 GiB.
- **Fracasso:** perde em todos os tamanhos, e o argumento fica só "um download e troca sem recarregar" (que continua valendo).
- Também sai a régua de sempre: bits/peso a mais que a curva dos k-quants interpolada, no mesmo tamanho de KL.

**Como mede.** Tudo no caminho F16: cada GGUF é desquantizado para F16 (`strata_escala_nivel.py`) e a KL sai do `llama-perplexity` contra o BF16, no mesmo caminho de ativações para todos (seções 27 a 30 de `docs/05`). A escada é refeita aqui com a mesma imatrix (os `_L` são o mesmo tipo com `token_embd` e `output` em Q8_0, como no bartowski), sobre o modelo base `unsloth/Meta-Llama-3.1-8B` dos notebooks anteriores. A mistura escolhe, enquanto couber, o passo (matriz, nível) com maior queda de Σ imatrix·(x − v)² por byte (`tests/demanda/strata_mistura.py`). O tamanho é o do Q4_K_M mais os planos (32 bytes por bloco por nível) e, nos tipos D, 16 bytes de tabela por matriz e nível, que é o que a RAM guarda com os tipos S.

**Limite conhecido.** A imatrix pesa as colunas pela ativação, não pelo efeito de cada camada na saída. Um alocador melhor pode mover a curva. Se o resultado ficar perto da linha, esse é o próximo teste.

In [ ]:
KLD_CHUNKS = 100           # trechos na divergência KL (~65 MB de disco por trecho)
IMATRIX_CHUNKS = 125       # trechos de 512 tokens do wikitext de treino para a imatrix
MODELO_HF = "unsloth/Meta-Llama-3.1-8B"
PASSO_GIB = 0.15           # distância entre os orçamentos da mistura
TIPOS = ['P', 'D']         # P = níveis de hoje (meio da célula); D = sub-células por tensor (seção 28)

In [ ]:
import os, re, json, shutil, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100 (ou L4).'
GPU = gpu.stdout.strip().splitlines()[0]
N = os.cpu_count()
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
DISCO = shutil.disk_usage('/content').free / 1e9
print(f'GPU: {GPU} | CPU: {N} threads | RAM {RAM:.0f} GB | disco livre {DISCO:.0f} GB')
if DISCO < 110:
    print('Aviso: o notebook precisa de ~100 GB de disco no pico. Ponha KLD_CHUNKS menor ou TIPOS = ["P"].')
if RAM < 50:
    print('Aviso: pouca RAM. Ligue a RAM alta.')

In [ ]:
STRATA_SHA = "1b0c37615b2d45533416dac833d2abb41b511a08"   # o commit do STRATA com tests/demanda/strata_mistura.py
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o commit em que o patch dos tipos foi feito
!rm -rf strata && git init -q strata && cd strata && git remote add origin https://github.com/Cantyll/strata.git && git fetch -q --depth 1 origin $STRATA_SHA && git checkout -q FETCH_HEAD
!rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin $LLAMA_SHA && git checkout -q FETCH_HEAD
!sh strata/kernel/ggml/aplicar.sh llama.cpp
!pip install -q -U transformers safetensors
!pip install -q -e llama.cpp/gguf-py
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON 2>&1 | tail -3 && cmake --build build -j $(nproc) --target llama-quantize llama-perplexity llama-imatrix 2>&1 | tail -5
!ls -la llama.cpp/build/bin | grep -E 'llama-(perplexity|quantize|imatrix)$'

In [ ]:
import urllib.request, zipfile, io
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
    open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
    open('/content/wiki.train.raw', 'wb').write(z.read('wikitext-2-raw/wiki.train.raw'))
    TEXTO = 'wikitext-2 raw'
except Exception as e:
    print('download do wikitext raw falhou:', e)
    base = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
    urllib.request.urlretrieve(base + 'test.txt', '/content/wiki.test.raw')
    urllib.request.urlretrieve(base + 'train.txt', '/content/wiki.train.raw')
    TEXTO = 'wikitext-2 (não-raw, cópia do PyTorch)'
print(TEXTO, os.path.getsize('/content/wiki.test.raw'), 'bytes de teste')

Modelo em BF16, imatrix, a escada do Hub refeita com a mesma imatrix, e a base da KL.

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
G = '/content/gguf'
os.makedirs(G, exist_ok=True)
!python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile {G}/BF16.gguf 2>&1 | tail -1
!rm -rf {src}
!llama.cpp/build/bin/llama-imatrix -m {G}/BF16.gguf -f /content/wiki.train.raw -ngl 99 -c 512 --chunks {IMATRIX_CHUNKS} -o {G}/imatrix.gguf > {G}/imatrix.log 2>&1
!tail -2 {G}/imatrix.log
# a escada do bartowski: os _L são o mesmo tipo com token_embd e output em Q8_0
ESCADA = {'Q4_K_M': 'Q4_K_M', 'Q4_K_L': 'Q4_K_M', 'Q5_K_S': 'Q5_K_S', 'Q5_K_M': 'Q5_K_M', 'Q5_K_L': 'Q5_K_M', 'Q6_K': 'Q6_K', 'Q6_K_L': 'Q6_K'}
for nome, q in ESCADA.items():
    extra = '--output-tensor-type q8_0 --token-embedding-type q8_0' if nome.endswith('_L') else ''
    !llama.cpp/build/bin/llama-quantize --imatrix {G}/imatrix.gguf {extra} {G}/BF16.gguf {G}/{nome}.gguf {q} > {G}/{nome}.log 2>&1
!llama.cpp/build/bin/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
!ls -la {G}/*.gguf /content/kld_base.bin
!df -h /content | tail -1

In [ ]:
kld, tam, planos = {}, {}, {}
def kl_de(path, nome, gib):
    out = !llama.cpp/build/bin/llama-perplexity -m {path} --kl-divergence-base /content/kld_base.bin --kl-divergence -ngl 99 2>&1
    txt = '\n'.join(out)
    k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
    top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
    if not k_:
        print(f'Erro na KL de {nome}:', '\n'.join(out[-8:]))
        return
    kld[nome] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
    tam[nome] = gib
    print(f'{nome}: KL ×1000 {1000 * kld[nome][0]:.2f}, mesmo 1º token {kld[nome][2]}%, {gib:.3f} GiB', flush=True)

ESC = f'python strata/tests/kernel/strata_escala_nivel.py --bf16 {G}/BF16.gguf --imatrix {G}/imatrix.gguf'
MIS = 'python strata/tests/demanda/strata_mistura.py'

A escada no caminho F16. O tamanho é o do GGUF de verdade; o F16 só serve para a KL.

In [ ]:
for nome in ESCADA:
    gib = os.path.getsize(f'{G}/{nome}.gguf') / 2**30
    !{ESC} --gguf {G}/{nome}.gguf --modo copia --out /content/e.gguf 2>&1 | grep -v Warning | tail -1
    kl_de('/content/e.gguf', nome, gib)
    if nome == 'Q4_K_M':
        !mv /content/e.gguf /content/e0.gguf
    else:
        !rm -f /content/e.gguf {G}/{nome}.gguf

As misturas. Para cada tipo: grava os níveis 1 e 2 em F16 (P: meio da célula; D: sub-células por tensor, a variante "borda" que o kernel D usa), mede o erro de cada matriz em cada nível e, para cada orçamento, grava a mistura e mede a KL. Também mede tudo no nível 1 e tudo no nível 2.

In [ ]:
import numpy as np
base = os.path.getsize(f'{G}/Q4_K_M.gguf')
for tipo in TIPOS:
    if tipo == 'P':
        !{ESC} --gguf {G}/Q4_K_M.gguf --modo fixo --out /content/e1.gguf /content/e2.gguf 2>&1 | grep -v Warning | tail -2
    else:
        !{ESC} --gguf {G}/Q4_K_M.gguf --modo desloc --grupo tensor --por borda --out /content/e1.gguf /content/e2.gguf 2>&1 | grep -v Warning | tail -2
    !{MIS} erros --bf16 {G}/BF16.gguf --imatrix {G}/imatrix.gguf --niveis /content/e0.gguf /content/e1.gguf /content/e2.gguf --out /content/erros_{tipo}.json > /content/erros_{tipo}.log 2>&1
    err = json.load(open(f'/content/erros_{tipo}.json'))
    tab = 16 if tipo == 'D' else 0
    blocos = sum(v['pesos'] for v in err.values()) // 256
    cheio = [(base + k * (blocos * 32 + tab * len(err))) / 2**30 for k in (1, 2)]
    orc = list(np.arange(base / 2**30 + PASSO_GIB, cheio[1] - 0.02, PASSO_GIB)) + cheio
    for g in sorted(orc):
        nome = f'{tipo}_{g:.2f}'
        out = !{MIS} grava --erros /content/erros_{tipo}.json --q4km {G}/Q4_K_M.gguf --gib {g + 1e-9} --tabela {tab} --niveis /content/e0.gguf /content/e1.gguf /content/e2.gguf --out /content/m.gguf --plano-out /content/plano_{nome}.json 2>&1
        print(out[-1])
        p = json.load(open(f'/content/plano_{nome}.json'))
        planos[nome] = {k: p[k] for k in ('gib', 'matrizes_por_nivel', 'fracao_pesos_por_nivel')}
        kl_de('/content/m.gguf', nome, p['gib'])
        !rm -f /content/m.gguf
    !rm -f /content/e1.gguf /content/e2.gguf

**A régua.**
- *melhor degrau que cabe*: o arquivo da escada com o maior tamanho que não passa do da mistura (o que a pessoa baixaria com essa RAM). *vence* = KL da mistura menor que a dele, fora da barra de erro (diferença > 2 ±).
- *bits/peso a mais*: quanto a mais a mistura gasta que a curva da escada interpolada (log da KL) com a mesma KL. Abaixo de 0, o STRATA ganha.
- *faixa ganha*: o maior trecho contínuo em que a mistura vence; cada ponto vale a largura do passo (0,15 GiB) em volta dele, então são precisos dois pontos vizinhos que vencem para chegar a 0,2 GiB.

In [ ]:
import math, gguf
from IPython.display import Markdown, display
r = gguf.GGUFReader(f'{G}/Q4_K_M.gguf')
n_cam = sum(int(t.n_elements) for t in r.tensors if t.name.startswith('blk.') and t.tensor_type.name in ('Q4_K', 'Q6_K') and len(t.shape) == 2)
esc = sorted((tam[q], kld[q][0], kld[q][1], q) for q in ESCADA if q in kld)

def curva_tam(k):
    # tamanho (GiB) da escada com KL k, interpolando o log da KL entre degraus vizinhos (fronteira: só os que melhoram)
    pts, best = [], float('inf')
    for s, kk, _, _ in esc:
        if kk < best:
            pts.append((s, kk)); best = kk
    for (s1, k1), (s2, k2) in zip(pts, pts[1:]):
        if k >= k2 or (s2, k2) == pts[-1]:
            if k > pts[0][1]:
                (s1, k1), (s2, k2) = pts[0], pts[1]
            return s1 + (s2 - s1) * math.log(k / k1) / math.log(k2 / k1)

def melhor_que_cabe(s):
    c = [(kk, e, q, ss) for ss, kk, e, q in esc if ss <= s + 1e-9]
    return min(c) if c else None

T = [f'# STRATA, E5: mistura de níveis contra a escada do Hub (Llama-3.1-8B, caminho F16)', '',
     f'GPU: {GPU}. KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos do {TEXTO} de teste; imatrix com {IMATRIX_CHUNKS} trechos de treino. '
     f'Matrizes das camadas: {n_cam / 1e9:.2f} bilhões de pesos.', '',
     '## Escada (refeita com a mesma imatrix)', '', '| arquivo | GiB | KL ×1000 | mesmo 1º token |', '|---|---:|---:|---:|']
for s, k, e, q in esc:
    T.append(f'| {q} | {s:.3f} | {1000 * k:.2f} ± {1000 * e:.2f} | {kld[q][2]:.2f}% |')
resumo = {}
for tipo in TIPOS:
    T += ['', f'## Mistura {tipo}', '',
          '| RAM (GiB) | matrizes 0/1/2 | pesos 0/1/2 | KL ×1000 | mesmo 1º token | melhor degrau que cabe | KL dele | vence? | bits/peso a mais |',
          '|---:|---|---|---:|---:|---|---:|---|---:|']
    pts = sorted((tam[n], n) for n in kld if n.startswith(tipo + '_'))
    ganhos = []
    for s, n in pts:
        k, e, top = kld[n]
        m = melhor_que_cabe(s)
        vence = m is not None and (m[0] - k) > 2 * math.hypot(e, m[1])
        ganhos.append((s, vence))
        st = curva_tam(k)
        b = (s - st) * 2**30 * 8 / n_cam if st is not None else float('nan')
        pl = planos.get(n, {})
        mp = '/'.join(map(str, pl.get('matrizes_por_nivel', [])))
        fp = '/'.join(f'{100 * x:.0f}' for x in pl.get('fracao_pesos_por_nivel', []))
        T.append(f'| {s:.3f} | {mp} | {fp}% | {1000 * k:.2f} ± {1000 * e:.2f} | {top:.2f}% | {m[2] if m else "-"} | '
                 f'{1000 * m[0]:.2f} | {"**sim**" if vence else "não"} | {b:+.3f} |' if m else f'| {s:.3f} | {mp} | {fp}% | {1000 * k:.2f} | {top:.2f}% | - | - | - | {b:+.3f} |')
    faixa, ini = 0.0, None
    for (s, v), prox in zip(ganhos, ganhos[1:] + [(None, False)]):
        if v and ini is None:
            ini = s
        if v and not prox[1]:
            faixa = max(faixa, s - ini + PASSO_GIB); ini = None   # cada ponto vale a largura do passo em volta dele
    resumo[tipo] = faixa
    T += ['', f'Faixa contínua ganha: **{faixa:.2f} GiB** (cada ponto que vence vale {PASSO_GIB} GiB em volta dele; um ponto só não chega à regra). '
          + ('**Sucesso pela regra (≥ 0,2 GiB).**' if faixa >= 0.2 - 1e-9 else '**Não chega à regra (≥ 0,2 GiB).**')]
os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_8b_mistura.md', 'w').write('\n'.join(T) + '\n')
json.dump({'texto': TEXTO, 'gpu': GPU, 'kld_chunks': KLD_CHUNKS, 'imatrix_chunks': IMATRIX_CHUNKS, 'kld': kld, 'gib': tam,
           'planos': planos, 'n_camadas': n_cam, 'faixa_ganha_gib': resumo}, open('/content/res/strata_8b_mistura.json', 'w'), indent=1)
display(Markdown('\n'.join(T)))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/strata_8b_mistura.md')
files.download('/content/res/strata_8b_mistura.json')